# 02 — Chạy dữ liệu không khử trùng lặp qua các model

**Câu hỏi:** vấn đề nằm ở *dữ liệu* hay ở *thuật toán train*?

**Thiết kế:** thuật toán giữ **nguyên tuyệt đối** — cùng cấu hình Random Forest của
run1/run2/run3, cùng tiền xử lý, cùng cách chia, cùng seed 42, cùng chỉ số.
Thứ duy nhất thay đổi là **bộ dữ liệu huấn luyện**. Đổi cả hai cùng lúc thì không
quy được trách nhiệm cho bên nào.

**Hai cách đánh giá — phải phân biệt rạch ròi:**

- **(A) Native** — đo trên tập test *không khử trùng lặp*. Đây là con số một nhóm sẽ
  nhận được nếu họ bỏ qua bước khử trùng lặp.
- **(B) Đối chứng** — đo trên **đúng 25.284 dòng `test-final` chính thức** mà mọi
  thực nghiệm trước đã dùng. Tập đánh giá giữ nguyên nên khác biệt duy nhất là dữ
  liệu huấn luyện. **Chỉ cách (B) trả lời được câu hỏi.**

Không ghi gì ra ngoài `experiment_nodedup/`.

In [1]:
import json, os, time
import numpy as np
import pandas as pd
import joblib

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.width', 250)
RS = 42
TARGETS = ['label', 'attack_cat']
CAT = ['proto', 'service', 'state']
VARIANT = 'V1_nodedup'

BASE = os.path.abspath(os.path.join('..', '..'))
NEW = os.path.join(BASE, 'experiment_nodedup')
ND = os.path.join(NEW, 'data', 'cleaned_nodedup')
OUTR, OUTM = os.path.join(NEW, 'reports'), os.path.join(NEW, 'models')

best_params = json.load(open(os.path.join(BASE, 'models', 'random_forest_v2_metrics.json'),
                             encoding='utf-8'))['best_params']
print('Siêu tham số run2 (GIỮ NGUYÊN, không dò lại):', best_params)


def feats(df):
    return df.drop(columns=TARGETS).astype({c: str for c in CAT})


def make_rf(**rf):
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT)],
                            remainder='passthrough', verbose_feature_names_out=False)
    return Pipeline([('preprocess', pre),
                     ('rf', RandomForestClassifier(**(dict(random_state=RS, n_jobs=-1) | rf)))])


def metrics_at(y_true, proba, threshold):
    y_true = np.asarray(y_true)
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'threshold': float(threshold), 'accuracy': (tp + tn) / len(pred),
            'precision': tp / max(tp + fp, 1), 'recall': tp / max(tp + fn, 1),
            'f1': 2 * tp / max(2 * tp + fp + fn, 1), 'fpr': fp / max(fp + tn, 1),
            'roc_auc': roc_auc_score(y_true, proba), 'pr_auc': average_precision_score(y_true, proba),
            'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)}


def threshold_at_fpr(y_true, proba, fpr_max=0.10):
    y_true = np.asarray(y_true)
    neg, pos = y_true == 0, y_true == 1
    best = (None, -1.0)
    for t in np.r_[np.unique(proba), 1.0 + 1e-9]:
        pred = proba >= t
        if (pred & neg).sum() / neg.sum() <= fpr_max:
            r = (pred & pos).sum() / pos.sum()
            if r > best[1]:
                best = (float(t), float(r))
    return best[0]

Siêu tham số run2 (GIỮ NGUYÊN, không dò lại): {'min_samples_leaf': 1, 'max_samples': 0.7, 'max_features': 0.5, 'max_depth': 20, 'class_weight': None}


In [2]:
nd_tr = pd.read_parquet(os.path.join(ND, f'{VARIANT}_UNSW_NB15_training-set.parquet'))
nd_te = pd.read_parquet(os.path.join(ND, f'{VARIANT}_UNSW_NB15_testing-set.parquet'))
Xnd, ynd = feats(nd_tr), nd_tr['label']
Xnd_tr, Xnd_val, ynd_tr, ynd_val = train_test_split(Xnd, ynd, test_size=0.2, stratify=ynd, random_state=RS)

dev_i, fin_i = train_test_split(nd_te.index, test_size=0.5, stratify=nd_te['attack_cat'], random_state=RS)
pd.DataFrame({'test_index': nd_te.index,
              'split': np.where(nd_te.index.isin(fin_i), 'test_final', 'test_dev')}
             ).to_csv(os.path.join(ND, f'{VARIANT}_test_split.csv'), index=False)
nd_dev, nd_fin = nd_te.loc[dev_i], nd_te.loc[fin_i]

# bộ chính thức: CHỈ ĐỌC, dùng làm tập đánh giá giữ-nguyên
of_tr = pd.read_parquet(os.path.join(BASE, 'data', 'cleaned', 'UNSW_NB15_training-set.parquet'))
of_te = pd.read_parquet(os.path.join(BASE, 'data', 'cleaned', 'UNSW_NB15_testing-set.parquet'))
of_sp = pd.read_csv(os.path.join(BASE, 'data', 'cleaned', 'test_split_v3.csv'))
of_fin = of_te.loc[of_sp.loc[of_sp['split'] == 'test_final', 'test_index'].values]
of_dev = of_te.loc[of_sp.loc[of_sp['split'] == 'test_dev', 'test_index'].values]
Xof, yof = feats(of_tr), of_tr['label']
Xof_tr, Xof_val, yof_tr, yof_val = train_test_split(Xof, yof, test_size=0.2, stratify=yof, random_state=RS)
Xof_fin, yof_fin = feats(of_fin), of_fin['label']
Xof_dev, yof_dev = feats(of_dev), of_dev['label']

display(pd.DataFrame({
    'train-fit': [len(Xnd_tr), len(Xof_tr)], 'validation': [len(Xnd_val), len(Xof_val)],
    'test-dev': [len(nd_dev), len(of_dev)], 'test-final': [len(nd_fin), len(of_fin)],
    'tỷ lệ Attack (train)': [ynd.mean(), yof.mean()],
}, index=['không khử trùng lặp', 'chính thức']).round(4))

,train-fit,validation,test-dev,test-final,tỷ lệ Attack (train)
không khử trùng lặp,136412,34103,40408,40408,0.6807
chính thức,79147,19787,25283,25284,0.4964


### Kiểm tra công bằng của phép so sánh

Nếu tập huấn luyện không-khử-trùng-lặp rò rỉ vào `test-final` chính thức nhiều hơn
hẳn, thì so sánh (B) sẽ thiên vị. Phải đo trước.

In [3]:
key = [c for c in of_tr.columns if c not in TARGETS] + ['label']
of_fin_keys = of_fin[key].astype(str).agg('|'.join, axis=1)


def keyset(df):
    return set(df[key].astype(str).agg('|'.join, axis=1))


leak = {'train-fit chính thức': float(of_fin_keys.isin(keyset(of_tr.loc[Xof_tr.index])).mean()),
        'train-fit không khử trùng lặp': float(of_fin_keys.isin(keyset(nd_tr.loc[Xnd_tr.index])).mean())}
print('Tỷ lệ dòng test-final CHÍNH THỨC cũng xuất hiện trong tập huấn luyện:')
for k, v in leak.items():
    print(f'  {k:32s} {v:.2%}')
print('\n=> Gần như bằng nhau, nên chênh lệch ở so sánh (B) KHÔNG phải do rò rỉ,')
print('   mà do số lần lặp của mỗi dòng (tức trọng số mẫu ngầm) đã thay đổi.')

Tỷ lệ dòng test-final CHÍNH THỨC cũng xuất hiện trong tập huấn luyện:
  train-fit chính thức             3.22%
  train-fit không khử trùng lặp    3.67%

=> Gần như bằng nhau, nên chênh lệch ở so sánh (B) KHÔNG phải do rò rỉ,
   mà do số lần lặp của mỗi dòng (tức trọng số mẫu ngầm) đã thay đổi.


## Giai đoạn M1 và M2 — đúng cấu hình của run1 và run2

In [4]:
STAGES = {'M1 cấu hình run1 (100 cây, không giới hạn độ sâu)': dict(n_estimators=100),
          'M2 cấu hình run2 (300 cây, đã tinh chỉnh)': dict(n_estimators=300, **best_params)}

eval_nd = {'nd validation': (Xnd_val, ynd_val),
           'nd test-dev': (feats(nd_dev), nd_dev['label']),
           'nd test-final': (feats(nd_fin), nd_fin['label'])}

rows, proba_store, fitted = {}, {}, {}
for train_name, (Xt, yt, esets) in {
    'train = KHÔNG KHỬ TRÙNG LẶP': (Xnd_tr, ynd_tr, eval_nd),
    'train = CHÍNH THỨC': (Xof_tr, yof_tr, {'of validation': (Xof_val, yof_val)}),
}.items():
    for sname, params in STAGES.items():
        t0 = time.time()
        m = make_rf(**params).fit(Xt, yt)
        tag = f'{sname} | {train_name}'
        fitted[tag] = m
        print(f'{tag}: fit {time.time() - t0:.0f}s')
        for ename, (Xe, ye) in {**esets, 'test-final CHÍNH THỨC': (Xof_fin, yof_fin)}.items():
            p = m.predict_proba(Xe)[:, 1]
            proba_store[f'{tag} || {ename}'] = p
            rows[f'{tag} || {ename}'] = metrics_at(ye, p, 0.5)
            r = rows[f'{tag} || {ename}']
            print(f'   {ename:24s} PR-AUC {r["pr_auc"]:.4f} | acc {r["accuracy"]:.4f} '
                  f'| prec {r["precision"]:.4f} | rec {r["recall"]:.4f} | fpr {r["fpr"]:.4f}')

M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = KHÔNG KHỬ TRÙNG LẶP: fit 13s
   nd validation            PR-AUC 0.9969 | acc 0.9601 | prec 0.9615 | rec 0.9807 | fpr 0.0837


   nd test-dev              PR-AUC 0.9802 | acc 0.8672 | prec 0.8203 | rec 0.9748 | fpr 0.2675
   nd test-final            PR-AUC 0.9789 | acc 0.8691 | prec 0.8216 | rec 0.9766 | fpr 0.2656


   test-final CHÍNH THỨC    PR-AUC 0.9240 | acc 0.8152 | prec 0.6771 | rec 0.9480 | fpr 0.2616


M2 cấu hình run2 (300 cây, đã tinh chỉnh) | train = KHÔNG KHỬ TRÙNG LẶP: fit 73s


   nd validation            PR-AUC 0.9974 | acc 0.9615 | prec 0.9625 | rec 0.9816 | fpr 0.0815


...

   nd test-final            PR-AUC 0.9846 | acc 0.8702 | prec 0.8229 | rec 0.9769 | fpr 0.2633
   test-final CHÍNH THỨC    PR-AUC 0.9387 | acc 0.8179 | prec 0.6800 | rec 0.9503 | fpr 0.2587


M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = CHÍNH THỨC: fit 9s
   of validation            PR-AUC 0.9815 | acc 0.9263 | prec 0.9020 | rec 0.9553 | fpr 0.1024


   test-final CHÍNH THỨC    PR-AUC 0.9265 | acc 0.8123 | prec 0.6728 | rec 0.9498 | fpr 0.2672


M2 cấu hình run2 (300 cây, đã tinh chỉnh) | train = CHÍNH THỨC: fit 49s
   of validation            PR-AUC 0.9849 | acc 0.9322 | prec 0.9104 | rec 0.9577 | fpr 0.0929


   test-final CHÍNH THỨC    PR-AUC 0.9414 | acc 0.8186 | prec 0.6813 | rec 0.9491 | fpr 0.2569


## Giai đoạn M3 — đúng cấu hình run3: isotonic calibration + ngưỡng FPR ≤ 10%

Ngưỡng luôn chọn trên `test-dev` của **chính** bộ dữ liệu huấn luyện tương ứng,
không bao giờ chọn trên tập đánh giá cuối.

In [5]:
for train_name, Xt, yt, Xd, yd, lbl in [
    ('train = KHÔNG KHỬ TRÙNG LẶP', Xnd_tr, ynd_tr, feats(nd_dev), nd_dev['label'], 'nd test-dev'),
    ('train = CHÍNH THỨC', Xof_tr, yof_tr, Xof_dev, yof_dev, 'test-dev chính thức'),
]:
    t0 = time.time()
    cal = CalibratedClassifierCV(make_rf(n_estimators=300, **best_params), method='isotonic', cv=5)
    cal.fit(Xt, yt)
    thr = threshold_at_fpr(yd, cal.predict_proba(Xd)[:, 1], 0.10)
    tag = f'M3 cấu hình run3 (calibrated, ngưỡng từ {lbl}) | {train_name}'
    fitted[tag] = (cal, thr)
    print(f'{tag}\n   fit {time.time() - t0:.0f}s | ngưỡng {thr:.4f}')
    targets = {'test-final CHÍNH THỨC': (Xof_fin, yof_fin)}
    if 'KHÔNG' in train_name:
        targets['nd test-final'] = (feats(nd_fin), nd_fin['label'])
    for ename, (Xe, ye) in targets.items():
        p = cal.predict_proba(Xe)[:, 1]
        proba_store[f'{tag} || {ename}'] = p
        rows[f'{tag} || {ename}'] = metrics_at(ye, p, thr)
        r = rows[f'{tag} || {ename}']
        print(f'   {ename:24s} PR-AUC {r["pr_auc"]:.4f} | acc {r["accuracy"]:.4f} '
              f'| prec {r["precision"]:.4f} | rec {r["recall"]:.4f} | fpr {r["fpr"]:.4f}')

M3 cấu hình run3 (calibrated, ngưỡng từ nd test-dev) | train = KHÔNG KHỬ TRÙNG LẶP
   fit 269s | ngưỡng 0.7636


   test-final CHÍNH THỨC    PR-AUC 0.9395 | acc 0.8827 | prec 0.8346 | rec 0.8481 | fpr 0.0973


   nd test-final            PR-AUC 0.9851 | acc 0.9162 | prec 0.9189 | rec 0.9315 | fpr 0.1029


M3 cấu hình run3 (calibrated, ngưỡng từ test-dev chính thức) | train = CHÍNH THỨC
   fit 185s | ngưỡng 0.7105


   test-final CHÍNH THỨC    PR-AUC 0.9418 | acc 0.8824 | prec 0.8265 | rec 0.8597 | fpr 0.1045


In [6]:
res = pd.DataFrame(rows).T
cols = ['threshold', 'pr_auc', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'fpr',
        'TN', 'FP', 'FN', 'TP']
display(res[cols].round(4))
res[cols].round(6).to_csv(os.path.join(OUTR, 'nodedup_results.csv'))
np.savez(os.path.join(OUTR, 'nodedup_proba.npz'), **proba_store)
joblib.dump(fitted, os.path.join(OUTM, 'nodedup_models.joblib'), compress=3)
print('đã lưu kết quả, xác suất và model vào experiment_nodedup/')

,threshold,pr_auc,roc_auc,accuracy,precision,recall,f1,fpr,TN,FP,FN,TP
"M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = KHÔNG KHỬ TRÙNG LẶP || nd validation",0.5000,0.9969,0.9942,0.9601,0.9615,0.9807,0.9710,0.0837,9977.0,911.0,449.0,22766.0
"M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = KHÔNG KHỬ TRÙNG LẶP || nd test-dev",0.5000,0.9802,0.9764,0.8672,0.8203,0.9748,0.8909,0.2675,13140.0,4799.0,567.0,21902.0
"M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = KHÔNG KHỬ TRÙNG LẶP || nd test-final",0.5000,0.9789,0.9757,0.8691,0.8216,0.9766,0.8924,0.2656,13175.0,4765.0,526.0,21942.0
"M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = KHÔNG KHỬ TRÙNG LẶP || test-final CHÍNH THỨC",0.5000,0.9240,0.9537,0.8152,0.6771,0.9480,0.7899,0.2616,11827.0,4190.0,482.0,8785.0
"M2 cấu hình run2 (300 cây, đã tinh chỉnh) | train = KHÔNG KHỬ TRÙNG LẶP || nd validation",0.5000,0.9974,0.9946,0.9615,0.9625,0.9816,0.9720,0.0815,10001.0,887.0,427.0,22788.0
"M2 cấu hình run2 (300 cây, đã tinh chỉnh) | train = KHÔNG KHỬ TRÙNG LẶP || nd test-dev",0.5000,0.9852,0.9798,0.8713,0.8246,0.9763,0.8940,0.2602,13272.0,4667.0,533.0,21936.0
"M2 cấu hình run2 (300 cây, đã tinh chỉnh) | train = KHÔNG KHỬ TRÙNG LẶP || nd test-final",0.5000,0.9846,0.9794,0.8702,0.8229,0.9769,0.8933,0.2633,13217.0,4723.0,520.0,21948.0
"M2 cấu hình run2 (300 cây, đã tinh chỉnh) | train = KHÔNG KHỬ TRÙNG LẶP || test-final CHÍNH THỨC",0.5000,0.9387,0.9584,0.8179,0.6800,0.9503,0.7927,0.2587,11873.0,4144.0,461.0,8806.0
"M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = CHÍNH THỨC || of validation",0.5000,0.9815,0.9841,0.9263,0.9020,0.9553,0.9279,0.1024,8945.0,1020.0,439.0,9383.0
"M1 cấu hình run1 (100 cây, không giới hạn độ sâu) | train = CHÍNH THỨC || test-final CHÍNH THỨC",0.5000,0.9265,0.9547,0.8123,0.6728,0.9498,0.7877,0.2672,11737.0,4280.0,465.0,8802.0


đã lưu kết quả, xác suất và model vào experiment_nodedup/


## Hai cách đọc đặt cạnh nhau

Bảng dưới là trọng tâm của cả thực nghiệm.

In [7]:
view = []
for sname in list(STAGES) + ['M3 cấu hình run3']:
    for tr_name in ['train = KHÔNG KHỬ TRÙNG LẶP', 'train = CHÍNH THỨC']:
        k_native = next((k for k in rows if k.startswith(sname) and tr_name in k
                         and k.endswith('nd test-final')), None)
        k_const = next((k for k in rows if k.startswith(sname) and tr_name in k
                        and k.endswith('test-final CHÍNH THỨC')), None)
        if k_const is None:
            continue
        view.append({
            'giai đoạn': sname.split(' (')[0],
            'dữ liệu train': 'không khử trùng lặp' if 'KHÔNG' in tr_name else 'chính thức',
            '(A) PR-AUC trên test không khử trùng lặp': rows[k_native]['pr_auc'] if k_native else np.nan,
            '(A) Accuracy': rows[k_native]['accuracy'] if k_native else np.nan,
            '(B) PR-AUC trên test-final chính thức': rows[k_const]['pr_auc'],
            '(B) Accuracy': rows[k_const]['accuracy'],
        })
display(pd.DataFrame(view).set_index(['giai đoạn', 'dữ liệu train']).round(4))

(A) PR-AUC trên test không khử trùng lặp  (A) Accuracy  (B) PR-AUC trên test-final chính thức  (B) Accuracy
giai đoạn        dữ liệu train                                                                                                                   
M1 cấu hình run1 không khử trùng lặp                                    0.9789        0.8691                                 0.9240        0.8152
                 chính thức                                                NaN           NaN                                 0.9265        0.8123
M2 cấu hình run2 không khử trùng lặp                                    0.9846        0.8702                                 0.9387        0.8179
                 chính thức                                                NaN           NaN                                 0.9414        0.8186
M3 cấu hình run3 không khử trùng lặp                                    0.9851        0.9162                                 0.9395        0.8827
                 chính thức                                                NaN           NaN                                 0.9418        0.8824

## Kiểm định: cùng 25.284 dòng đánh giá, chỉ khác dữ liệu huấn luyện

In [8]:
BOOT = ['pr_auc', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'fpr']


def _pt(y_true, p, thr):
    pred = p >= thr; pos = y_true == 1
    tp = np.count_nonzero(pred & pos); fp = np.count_nonzero(pred & ~pos)
    fn = np.count_nonzero(~pred & pos); tn = np.count_nonzero(~pred & ~pos)
    return {'pr_auc': average_precision_score(y_true, p), 'roc_auc': roc_auc_score(y_true, p),
            'accuracy': (tp + tn) / len(y_true), 'precision': tp / max(tp + fp, 1),
            'recall': tp / max(tp + fn, 1), 'f1': 2 * tp / max(2 * tp + fp + fn, 1),
            'fpr': fp / max(fp + tn, 1)}


def paired_bootstrap(y_true, models, B=1000, seed=RS):
    y_true = np.asarray(y_true); n = len(y_true)
    rng = np.random.default_rng(seed)
    draws = {k: np.full((B, len(BOOT)), np.nan) for k in models}
    for b in range(B):
        idx = rng.integers(0, n, n); yb = y_true[idx]
        if yb.min() == yb.max():
            continue
        for k, (p, thr) in models.items():
            r = _pt(yb, np.asarray(p)[idx], thr)
            draws[k][b] = [r[m] for m in BOOT]
    return draws


def diff_ci(draws, a, b):
    d = draws[a] - draws[b]
    out = {}
    for i, m in enumerate(BOOT):
        col = d[:, i][~np.isnan(d[:, i])]
        lo, hi = np.percentile(col, 2.5), np.percentile(col, 97.5)
        out[m] = {'hiệu_TB': float(col.mean()), 'CI_thấp': float(lo), 'CI_cao': float(hi),
                  'loại_trừ_0': bool(lo > 0 or hi < 0),
                  'p_hai_phía': float(2 * min((col <= 0).mean(), (col >= 0).mean()))}
    return pd.DataFrame(out).T


for sname in STAGES:
    a = f'{sname} | train = KHÔNG KHỬ TRÙNG LẶP || test-final CHÍNH THỨC'
    b = f'{sname} | train = CHÍNH THỨC || test-final CHÍNH THỨC'
    dr = paired_bootstrap(yof_fin, {'nodedup': (proba_store[a], 0.5), 'official': (proba_store[b], 0.5)})
    print(f'--- {sname}: (không khử trùng lặp) − (chính thức) ---')
    display(diff_ci(dr, 'nodedup', 'official').round(5))

--- M1 cấu hình run1 (100 cây, không giới hạn độ sâu): (không khử trùng lặp) − (chính thức) ---


,hiệu_TB,CI_thấp,CI_cao,loại_trừ_0,p_hai_phía
pr_auc,-0.002432,-0.004544,-0.000333,True,0.026
roc_auc,-0.000944,-0.001739,-0.000126,True,0.026
accuracy,0.0029,0.000435,0.005418,True,0.028
precision,0.004234,0.001245,0.007233,True,0.006
recall,-0.001742,-0.004705,0.001727,False,0.322
f1,0.002289,-0.000303,0.004851,False,0.076
fpr,-0.005585,-0.009172,-0.002139,True,0.002


--- M2 cấu hình run2 (300 cây, đã tinh chỉnh): (không khử trùng lặp) − (chính thức) ---


,hiệu_TB,CI_thấp,CI_cao,loại_trừ_0,p_hai_phía
pr_auc,-0.002706,-0.003743,-0.001668,True,0.0
roc_auc,-0.001016,-0.00152,-0.00051,True,0.0
accuracy,-0.000759,-0.002809,0.001385,False,0.498
precision,-0.001312,-0.003869,0.001251,False,0.322
recall,0.001158,-0.001292,0.003654,False,0.384
f1,-0.000486,-0.002633,0.001621,False,0.686
fpr,0.001868,-0.001065,0.004942,False,0.222


## Kết luận

**Vấn đề KHÔNG nằm ở thuật toán train.** Thuật toán giữ nguyên hoàn toàn ở cả hai
nhánh; khi đo trên cùng một tập đánh giá trung thực, hai nhánh cho kết quả gần như
y hệt nhau.

**Giữ lại dòng trùng không sửa được gì.** Trên tập `test-final` chính thức, model
huấn luyện bằng dữ liệu không khử trùng lặp **kém hơn một chút**, chứ không tốt hơn.
Số lần lặp của mỗi dòng chỉ đóng vai trò như trọng số mẫu ngầm, và trọng số đó
không chữa được việc phân bố train khác phân bố test.

**Con số đẹp của cách đánh giá (A) là ảo.** PR-AUC ~0,98 và Accuracy ~0,87 trên tập
test không khử trùng lặp đến từ việc khoảng 26% dòng test là bản sao của dòng train.
Đó là thuộc lòng, không phải tổng quát hoá.

**Trần hiệu năng nằm ở dữ liệu**, nhưng là ở *phân bố* train khác test, không phải ở
việc đã xoá dòng trùng.